# Step 4b — Build New Feature Blocks & Re-generate Datasets (v2)

## Objective
Add two new feature blocks to the existing baseline features, following the same **leakage-free protocol** (only data at or before $t_0$):

| Block | Features | Source |
|-------|----------|--------|
| **Treatment** | `riluzole_pre_t0` (0/1), `study_arm` (Active/Placebo), `n_conmeds_pre_t0` (count) | PROACT_RILUZOLE, PROACT_TREATMENT, PROACT_CONMEDS |
| **Labs** | `creatinine_t0`, `alt_t0` (last pre-baseline values) | PROACT_LABS |

## Outputs
- `01_data/processed/features_treatment_t0.csv` — treatment block (1 row per subject)
- `01_data/processed/features_labs_t0.csv` — labs block (1 row per subject)
- `01_data/processed/dataset_6m_v2.csv` — full merged dataset for 6-month horizon
- `01_data/processed/dataset_3m_v2.csv` — full merged dataset for 3-month horizon
- `04_outputs/tables/step4_featureblock_coverage_v2.csv` — updated coverage table

## Temporal rule
> All features use only records with `delta ≤ t0`. For longitudinal tables, we select the **last pre-baseline measurement** per subject.

In [ ]:
import os
import numpy as np
import pandas as pd

RAW       = os.path.join("..", "01_data", "raw")
INTERIM   = os.path.join("..", "01_data", "interim")
PROCESSED = os.path.join("..", "01_data", "processed")
OUT_TABLES = os.path.join("..", "04_outputs", "tables")

os.makedirs(PROCESSED, exist_ok=True)
os.makedirs(OUT_TABLES, exist_ok=True)

# --- Load baseline (anchor for t0) ---
base = pd.read_csv(os.path.join(INTERIM, "baseline_table_ALSFRS_R.csv"))
print("Baseline subjects:", base["subject_id"].nunique())

# --- Load existing feature blocks (from Step 4a) ---
vitals_feat = pd.read_csv(os.path.join(PROCESSED, "features_vitals_t0.csv"))
fvc_feat    = pd.read_csv(os.path.join(PROCESSED, "features_fvc_t0.csv"))

# --- Load targets (from Step 3) ---
targets = pd.read_csv(os.path.join(INTERIM, "baseline_with_targets_step3_nolabelsv2.csv"))

# --- Load new raw sources ---
riluzole  = pd.read_csv(os.path.join(RAW, "PROACT_RILUZOLE.csv"))
treatment = pd.read_csv(os.path.join(RAW, "PROACT_TREATMENT.csv"))
conmeds   = pd.read_csv(os.path.join(RAW, "PROACT_CONMEDS.csv"))
labs      = pd.read_csv(os.path.join(RAW, "PROACT_LABS.csv"))

print(f"Riluzole: {riluzole.shape}, Treatment: {treatment.shape}")
print(f"ConMeds: {conmeds.shape}, Labs: {labs.shape}")

## 1) Treatment Block

### Riluzole (`riluzole_pre_t0`)
Binary flag: did the subject use riluzole at or before baseline?
- If `Riluzole_use_Delta ≤ t0` and `Subject_used_Riluzole == "Yes"` → 1, else 0.
- Subjects not in the riluzole table are coded as 0 (no riluzole data = no usage evidence).

### Study Arm (`study_arm`)
Categorical: "Active" or "Placebo". Captures whether the subject was in a drug trial arm.
- Subjects not in the treatment table get NaN (will be imputed downstream as a separate category).

### Number of Concomitant Medications (`n_conmeds_pre_t0`)
Integer count of unique medications started at or before t0.
- Proxy for comorbidity burden.
- Subjects not in ConMeds → 0.

In [ ]:
# --- Helper: get t0 per subject ---
t0_map = dict(zip(base["subject_id"], base["t0_delta_days"]))

# ===== 1a) Riluzole =====
ril = riluzole.copy()
ril["t0"] = ril["subject_id"].map(t0_map)
# Keep only pre-baseline riluzole use
ril_pre = ril[(ril["Subject_used_Riluzole"] == "Yes") & (ril["Riluzole_use_Delta"] <= ril["t0"])]
ril_subjects = set(ril_pre["subject_id"].unique())

# ===== 1b) Study Arm =====
treat = treatment.drop_duplicates(subset="subject_id", keep="first")  # 1 row per subject
treat_feat = treat[["subject_id", "Study_Arm"]].rename(columns={"Study_Arm": "study_arm"})

# ===== 1c) Number of ConMeds pre-baseline =====
cm = conmeds.copy()
cm["t0"] = cm["subject_id"].map(t0_map)
cm["Start_Delta"] = pd.to_numeric(cm["Start_Delta"], errors="coerce")
cm_pre = cm.dropna(subset=["Start_Delta", "t0"])
cm_pre = cm_pre[cm_pre["Start_Delta"] <= cm_pre["t0"]]
n_conmeds = cm_pre.groupby("subject_id")["Medication_Coded"].nunique().reset_index()
n_conmeds.columns = ["subject_id", "n_conmeds_pre_t0"]

# ===== Build treatment feature table =====
treatment_feat = base[["subject_id"]].copy()
treatment_feat["riluzole_pre_t0"] = treatment_feat["subject_id"].isin(ril_subjects).astype(int)
treatment_feat = treatment_feat.merge(treat_feat, on="subject_id", how="left")
treatment_feat = treatment_feat.merge(n_conmeds, on="subject_id", how="left")
treatment_feat["n_conmeds_pre_t0"] = treatment_feat["n_conmeds_pre_t0"].fillna(0).astype(int)

out_treat = os.path.join(PROCESSED, "features_treatment_t0.csv")
treatment_feat.to_csv(out_treat, index=False)

print(f"Treatment features saved: {out_treat} | shape: {treatment_feat.shape}")
print(f"  riluzole_pre_t0: {treatment_feat['riluzole_pre_t0'].mean():.1%} = Yes")
print(f"  study_arm distribution:\n{treatment_feat['study_arm'].value_counts(dropna=False)}")
print(f"  n_conmeds_pre_t0: mean={treatment_feat['n_conmeds_pre_t0'].mean():.1f}, "
      f"median={treatment_feat['n_conmeds_pre_t0'].median():.0f}")
treatment_feat.head()

## 2) Labs Block

### Selection rationale
From the literature (Qin et al. 2022, Pancotti et al. 2022), **creatinine** is a biomarker of muscle mass that correlates with motor neuron loss, and **ALT** reflects hepatic function and general metabolic state.

Both have adequate coverage in the 6m cohort (~65–69%) and capture biologically distinct signals.

### Temporal rule
Same as vitals/FVC: select the **last pre-baseline measurement** (`Laboratory_Delta ≤ t0`) per subject per test.

In [ ]:
# ===== Extract last pre-baseline lab value per subject per test =====
LAB_TESTS = {
    "Creatinine": "creatinine_t0",
    "ALT(SGPT)":  "alt_t0",
}

labs_sub = labs[labs["Test_Name"].isin(LAB_TESTS.keys())].copy()
labs_sub["Laboratory_Delta"] = pd.to_numeric(labs_sub["Laboratory_Delta"], errors="coerce")
labs_sub["Test_Result"]      = pd.to_numeric(labs_sub["Test_Result"], errors="coerce")
labs_sub["t0"] = labs_sub["subject_id"].map(t0_map)

# Keep only pre-baseline
labs_sub = labs_sub.dropna(subset=["Laboratory_Delta", "t0", "Test_Result"])
labs_sub = labs_sub[labs_sub["Laboratory_Delta"] <= labs_sub["t0"]]

# Last pre-baseline per subject × test
labs_sub = labs_sub.sort_values(["subject_id", "Test_Name", "Laboratory_Delta"])
labs_last = labs_sub.groupby(["subject_id", "Test_Name"], as_index=False).tail(1)

# Pivot to wide format (1 row per subject)
labs_wide = labs_last.pivot(index="subject_id", columns="Test_Name", values="Test_Result")
labs_wide = labs_wide.rename(columns=LAB_TESTS).reset_index()

# Merge onto baseline subjects
labs_feat = base[["subject_id"]].merge(labs_wide, on="subject_id", how="left")

out_labs = os.path.join(PROCESSED, "features_labs_t0.csv")
labs_feat.to_csv(out_labs, index=False)

print(f"Labs features saved: {out_labs} | shape: {labs_feat.shape}")
for col, name in LAB_TESTS.items():
    n = labs_feat[name].notna().sum()
    print(f"  {name}: {n}/{len(labs_feat)} ({100*n/len(labs_feat):.1f}%) non-null")
    print(f"    mean={labs_feat[name].mean():.2f}, median={labs_feat[name].median():.2f}")
labs_feat.head()

## 3) Merge all feature blocks → final datasets v2

Merge order (all left-joins onto baseline):
1. **Baseline** (demographics, ALSFRS-R items) — 100% coverage
2. **Vitals** (weight, height, BMI, pulse, BP, ...) — ~82%
3. **FVC** (best trial liters, % normal) — ~53%
4. **Treatment** (riluzole, study_arm, n_conmeds) — 68–91% ← NEW
5. **Labs** (creatinine, ALT) — 65–69% ← NEW

Then filter to subjects with valid slopes for each horizon:
- `dataset_3m_v2.csv`: subjects with `slope_90d_per_30d` not NaN
- `dataset_6m_v2.csv`: subjects with `slope_180d_per_30d` not NaN

In [ ]:
# ===== Full merge: baseline + vitals + FVC + treatment + labs =====
full = (
    base
    .merge(vitals_feat, on="subject_id", how="left")
    .merge(fvc_feat,    on="subject_id", how="left")
    .merge(treatment_feat.drop(columns=["subject_id"], errors="ignore"),
           left_index=True, right_index=True, how="left")  # same order as base
)
# Re-merge treatment properly via subject_id
full = (
    base
    .merge(vitals_feat,    on="subject_id", how="left")
    .merge(fvc_feat,       on="subject_id", how="left")
    .merge(treatment_feat, on="subject_id", how="left")
    .merge(labs_feat,      on="subject_id", how="left")
)

# Add slopes from targets
slope_cols = ["subject_id", "slope_90d_per_30d", "slope_180d_per_30d"]
slope_cols = [c for c in slope_cols if c in targets.columns]
full = full.merge(targets[slope_cols], on="subject_id", how="left")

print(f"Full merged table: {full.shape}")
print(f"Columns: {list(full.columns)}")

# ===== Filter to horizon-specific datasets =====
df_6m = full.dropna(subset=["slope_180d_per_30d"]).copy()
df_3m = full.dropna(subset=["slope_90d_per_30d"]).copy()

# Save v2 datasets
out_6m = os.path.join(PROCESSED, "dataset_6m_v2.csv")
out_3m = os.path.join(PROCESSED, "dataset_3m_v2.csv")
df_6m.to_csv(out_6m, index=False)
df_3m.to_csv(out_3m, index=False)

print(f"\ndataset_6m_v2: {df_6m.shape} → {out_6m}")
print(f"dataset_3m_v2: {df_3m.shape} → {out_3m}")

## 4) Updated coverage table

Document coverage per feature block for both horizons.
This table will be used in the thesis to justify feature inclusion/exclusion decisions.

In [ ]:
# ===== Coverage analysis per block, per horizon =====
def coverage_report(df, horizon_label):
    N = len(df)
    rows = []
    # Vitals: at least weight or pulse present
    n_vit = df[["Weight_kg_t0", "Pulse"]].notna().any(axis=1).sum()
    rows.append(("Vitals", n_vit, n_vit / N * 100))
    # FVC
    n_fvc = df["FVC_Liters_best_t0"].notna().sum()
    rows.append(("FVC", n_fvc, n_fvc / N * 100))
    # Treatment (riluzole is always filled; study_arm may be NaN)
    n_arm = df["study_arm"].notna().sum()
    rows.append(("Treatment (study_arm)", n_arm, n_arm / N * 100))
    n_ril = df["riluzole_pre_t0"].sum()
    rows.append(("Treatment (riluzole=Yes)", int(n_ril), n_ril / N * 100))
    # Labs
    for col in ["creatinine_t0", "alt_t0"]:
        n = df[col].notna().sum()
        rows.append((f"Labs ({col})", n, n / N * 100))

    cov = pd.DataFrame(rows, columns=["block", f"n_{horizon_label}", f"pct_{horizon_label}"])
    return cov

cov_6m = coverage_report(df_6m, "6m")
cov_3m = coverage_report(df_3m, "3m")

cov_all = cov_6m.merge(cov_3m, on="block", how="outer")
print(f"\n=== Feature Block Coverage ===")
print(cov_all.to_string(index=False))

out_cov = os.path.join(OUT_TABLES, "step4_featureblock_coverage_v2.csv")
cov_all.to_csv(out_cov, index=False)
print(f"\nSaved: {out_cov}")

## 5) Re-generate held-out splits for v2 datasets

The held-out split is based on `subject_id`, which is unchanged between v1 and v2 (same subjects, just more columns). We re-use the **same split files** generated in Step 0 — no need to regenerate.

Let's verify that all v2 subjects are covered by the existing splits.

In [ ]:
# Verify that held-out splits cover all v2 subjects
split_6m = pd.read_csv(os.path.join(PROCESSED, "holdout_split_6m.csv"))
split_3m = pd.read_csv(os.path.join(PROCESSED, "holdout_split_3m.csv"))

assert set(df_6m["subject_id"]) == set(split_6m["subject_id"]), "6m subject mismatch!"
assert set(df_3m["subject_id"]) == set(split_3m["subject_id"]), "3m subject mismatch!"

print("✓ Held-out splits match v2 datasets perfectly.")
print(f"  6m: {len(split_6m[split_6m['partition']=='dev'])} dev + "
      f"{len(split_6m[split_6m['partition']=='test'])} test = {len(split_6m)}")
print(f"  3m: {len(split_3m[split_3m['partition']=='dev'])} dev + "
      f"{len(split_3m[split_3m['partition']=='test'])} test = {len(split_3m)}")

# Quick summary of new features in v2
print(f"\n=== v2 dataset summary ===")
print(f"6m: {df_6m.shape[0]} subjects × {df_6m.shape[1]} columns")
print(f"3m: {df_3m.shape[0]} subjects × {df_3m.shape[1]} columns")
new_cols = [c for c in df_6m.columns if c not in pd.read_csv(os.path.join(PROCESSED, "dataset_6m_v1.csv"), nrows=0).columns]
print(f"New columns in v2: {new_cols}")